In [1]:
import json
import time
import configparser
from datetime import datetime, timedelta
from Bio import Entrez
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk import download
from nltk.corpus import stopwords

download('punkt_tab')
download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Huawei\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Huawei\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
class GettingArticles:
    def __init__(self, email: str, api_key: str, topic_name: str, term_base: str, retmax: int = 150, max_articles = 100):
        Entrez.email = email
        Entrez.api_key = api_key
        self.term_base = term_base
        self.topic_name = topic_name
        self.retmax = retmax
        self.max_articles = max_articles
        self.stop_words = set(stopwords.words('english'))

    def download_interval(self, datetime_from: datetime, datetime_to: datetime):
        str_from = datetime_from.strftime("%Y/%m/%d")
        str_to = datetime_to.strftime("%Y/%m/%d")

        term_with_date = f"({self.term_base}) AND ({str_from}:{str_to}[dp])"
        print(f"{self.topic_name} for the period from {str_from} to {str_to}.")

        return self.search_and_download(term_with_date)

    def preprocess(self, term: str) -> list:
        term = term.lower()
        tokens = word_tokenize(term)
        no_punkt = [
            word for word in tokens 
            if word.isalpha() and '.' not in word and len(word) > 1
        ]
        clean_tokens = [word for word in no_punkt if word not in self.stop_words]
        return clean_tokens

    def search_and_download(self, term):
        handle = Entrez.esearch(db='pmc', term=term, retmax=self.retmax)
        record = Entrez.read(handle)
        handle.close()

        time.sleep(0.1)

        id_list = record["IdList"]

        if not id_list:
            return [], []

        fetch = Entrez.efetch(db="pmc", id=",".join(id_list), retmode="xml")
        xml_data = fetch.read()
        fetch.close()

        time.sleep(0.1)

        soup = BeautifulSoup(xml_data, "xml")
        articles = soup.find_all("article")

        raw_articles = []
        tokenized_articles = []

        for _, article in enumerate(articles):
            if len(tokenized_articles) == self.max_articles:
                break

            title_tag = article.find("article-title")
            title = title_tag.get_text().strip() if title_tag else "No title"

            abstract_tag = article.find("abstract")
            if abstract_tag:
                for title_in_abstract in abstract_tag.find_all("title"):
                    title_in_abstract.decompose()
                    
                abstract = abstract_tag.get_text().strip()
            else:
                abstract = "No abstract"

            body = article.find("body")
            full_text = ""

            if body:
                bad_tags = ["xref", "boxed-text", "fig"]
                for tag_name in bad_tags:
                    for tag in body.find_all(tag_name):
                        tag.decompose()
                        
                paragraphs = []
                for p in body.find_all("p"):
                    p_text = p.get_text().strip()
                    if p_text:
                        paragraphs.append(p_text)

                full_text = "\n\n".join(paragraphs)
            else:
                continue

            raw_articles.append({
                "title": title,
                "abstract": abstract,
                "text": full_text
            })

            title_tokens = self.preprocess(title) if title != "No title" else []
            abstract_tokens = self.preprocess(abstract) if abstract != "No abstract" else []
            text_tokens = self.preprocess(full_text)

            tokenized_articles.append({
                "title_tokens": title_tokens,
                "abstract_tokens": abstract_tokens,
                "text_tokens": text_tokens
            })

        return raw_articles, tokenized_articles

config = configparser.ConfigParser()
config.read('config.ini')

email = config['ENTREZ']['email']
api_key = config['ENTREZ']['api_key']

days_back = config.getint('SCRAPER', 'days_back')
retmax = config.getint('SCRAPER', 'retmax')
max_articles = config.getint('SCRAPER', 'max_articles')
queries = config['QUERIES']

date_to = datetime.now()
date_from = date_to - timedelta(days=days_back)

all_raw_articles = {}
all_tokenized_articles = {}

for topic, query in queries.items():
    scraper = GettingArticles(
        email=email,
        api_key=api_key,
        topic_name=topic,
        term_base=query,
        retmax=retmax,
        max_articles=max_articles
    )
    raw, tokenized = scraper.download_interval(date_from, date_to)

    all_raw_articles[topic] = raw
    all_tokenized_articles[topic] = tokenized

with open("med_psycho_neuro_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_raw_articles, f, ensure_ascii=False, indent=4)

with open("med_psycho_neuro_tokenized.json", "w", encoding="utf-8") as f:
    json.dump(all_tokenized_articles, f, ensure_ascii=False, indent=4)

medicine for the period from 2026/10/20 to 2026/11/19.
psychology for the period from 2026/10/20 to 2026/11/19.
neuroscience for the period from 2026/10/20 to 2026/11/19.
